## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [2]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [4]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        self.W1 = tf.Variable(tf.random.normal([28 * 28, 100], stddev=0.1), name='W1')  # 输入层到隐藏层权重
        self.b1 = tf.Variable(tf.zeros([100]), name='b1')  # 隐藏层偏置
        self.W2 = tf.Variable(tf.random.normal([100,100]), name='W2')
        self.b2 = tf.Variable(tf.zeros([100], name='b2'))
        self.W3 = tf.Variable(tf.random.normal([100,100]), name='W3')
        self.b3 = tf.Variable(tf.zeros([100], name='b3')                              )
        self.W4 = tf.Variable(tf.random.normal([100, 10], stddev=0.1), name='W4')  # 隐藏层到输出层权重
        self.b4 = tf.Variable(tf.zeros([10]), name='b4')  # 输出层偏置
        ####################
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        x = tf.reshape(x, [-1, 28 * 28])  # 将图像拉平成向量
        h1 = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)  # 第一层全连接并接 ReLU
        h2 = tf.nn.relu(tf.matmul(h1, self.W2) + self.b2) # 第二层
        h3 = tf.nn.relu(tf.matmul(h2, self.W3) + self.b3) # 第三层
        logits = tf.matmul(h3, self.W4) + self.b4  # 输出未归一化的 logits
        ####################
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [5]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [7]:
train_data, test_data = mnist_dataset()
for epoch in range(100):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 2.1765432 ; accuracy 0.7111
epoch 1 : loss 2.1516247 ; accuracy 0.71393335
epoch 2 : loss 2.1274192 ; accuracy 0.71613336
epoch 3 : loss 2.1038942 ; accuracy 0.71816665
epoch 4 : loss 2.080949 ; accuracy 0.7202833
epoch 5 : loss 2.058624 ; accuracy 0.72215
epoch 6 : loss 2.0369663 ; accuracy 0.7241333
epoch 7 : loss 2.015896 ; accuracy 0.7259833
epoch 8 : loss 1.9953625 ; accuracy 0.72796667
epoch 9 : loss 1.9753873 ; accuracy 0.73
epoch 10 : loss 1.9559581 ; accuracy 0.73225
epoch 11 : loss 1.9370526 ; accuracy 0.7338833
epoch 12 : loss 1.9186206 ; accuracy 0.73586667
epoch 13 : loss 1.900719 ; accuracy 0.73755
epoch 14 : loss 1.8832759 ; accuracy 0.73945
epoch 15 : loss 1.866234 ; accuracy 0.74093336
epoch 16 : loss 1.8496106 ; accuracy 0.74245
epoch 17 : loss 1.8333508 ; accuracy 0.74403334
epoch 18 : loss 1.8174859 ; accuracy 0.7456
epoch 19 : loss 1.8020054 ; accuracy 0.7470833
epoch 20 : loss 1.7868601 ; accuracy 0.7486333
epoch 21 : loss 1.7719914 ; accuracy 0.750